In [67]:
import requests
from bs4 import BeautifulSoup

# URL de la página de resultados de fútbol de ESPN
url ="https://www.espn.com.mx/futbol/resultados/_/liga/eng.1"
# Realizar la solicitud HTTP
response = requests.get(url)

# Verificar si la solicitud fue exitosa
if response.status_code == 200:
    # Parsear el contenido HTML
    soup = BeautifulSoup(response.text, 'html.parser')
    
    # Encontrar todos los contenedores de partidos
    partidos = soup.find_all('div', class_='ScoreboardScoreCell__Competitors')
    
    # Iterar sobre cada partido y extraer la información
    for partido in partidos:
        # Extraer los nombres de los equipos
        equipos = partido.find_all('div', class_='ScoreCell__TeamName')
        equipo1 = equipos[0].text.strip()
        equipo2 = equipos[1].text.strip()
        
        # Extraer los resultados
        resultados = partido.find_all('div', class_='ScoreCell__Score')
        resultado1 = resultados[0].text.strip()
        resultado2 = resultados[1].text.strip()
        
        # Imprimir la información del partido
        print(f"{equipo1} {resultado1} - {resultado2} {equipo2}")
else:
    print(f"Error al acceder al sitio: {response.status_code}")

Error al acceder al sitio: 403


In [12]:
import requests
import pandas as pd

API_KEY = "d527519e11msh7ce963ccbe620b9p10158ejsn257bb56b98a2"
API_HOST = "api-football-v1.p.rapidapi.com"

def get_live_matches():
    url = "https://api-football-v1.p.rapidapi.com/v3/fixtures"
    querystring = {"live":"all"}
    headers = {
        "X-RapidAPI-Key": API_KEY,
        "X-RapidAPI-Host": API_HOST
    }
    
    response = requests.get(url, headers=headers, params=querystring)
    return response.json()

def get_match_events(fixture_id):
    url = "https://api-football-v1.p.rapidapi.com/v3/fixtures/events"
    querystring = {"fixture": fixture_id}
    headers = {
        "X-RapidAPI-Key": API_KEY,
        "X-RapidAPI-Host": API_HOST
    }
    
    response = requests.get(url, headers=headers, params=querystring)
    return response.json()

try:
    # Obtener partidos en curso
    live_data = get_live_matches()
    
    if live_data['response']:
        matches_info = []
        
        for fixture in live_data['response']:
            fixture_id = fixture['fixture']['id']
            events_data = get_match_events(fixture_id)
            
            # Contar tarjetas rojas
            red_cards = {
                'home': 0,
                'away': 0
            }
            
            if 'response' in events_data:
                for event in events_data['response']:
                    if event['type'] == 'Card' and event['detail'] == 'Red Card':
                        if event['team']['id'] == fixture['teams']['home']['id']:
                            red_cards['home'] += 1
                        else:
                            red_cards['away'] += 1
            
            match_info = {
                'Liga': fixture['league']['name'],
                'Local': fixture['teams']['home']['name'],
                'Visitante': fixture['teams']['away']['name'],
                'Marcador': f"{fixture['goals']['home']}-{fixture['goals']['away']}",
                'Estado': fixture['fixture']['status']['long'],
                'Tarjetas Rojas (Local)': red_cards['home'],
                'Tarjetas Rojas (Visitante)': red_cards['away'],
                'Minuto': fixture['fixture']['status']['elapsed']
            }
            matches_info.append(match_info)
        
        # Mostrar resultados
        df = pd.DataFrame(matches_info)
        print("\n⚽ Partidos en vivo con estadísticas de tarjetas rojas:")
        print(df[['Liga', 'Local', 'Marcador', 'Visitante', 'Tarjetas Rojas (Local)', 'Tarjetas Rojas (Visitante)', 'Minuto']])
        
        # Guardar en CSV
        df.to_csv("partidos_con_tarjetas.csv", index=False)
        print("\n✅ Datos guardados en 'partidos_con_tarjetas.csv'")
    else:
        print("No hay partidos en curso en este momento.")

except Exception as e:
    print(f"Error: {e}")

pd.read_csv("partidos_con_tarjetas.csv")



⚽ Partidos en vivo con estadísticas de tarjetas rojas:
                                   Liga                     Local Marcador  \
0      Tercera División RFEF - Group 12                    Ibarra      1-1   
1      Tercera División RFEF - Group 12               Unión Viera      0-0   
2      Tercera División RFEF - Group 12           UD San Fernando      2-0   
3                   Division Intermedia      Independiente F.b.c.      1-0   
4       Division Profesional - Apertura         Libertad Asuncion      0-0   
5                      Primera División  Estudiantes de Merida FC      0-0   
6                              Liga Pro                     Aucas      0-0   
7                      Primera División            Union Espanola      0-1   
8                             Primera B                  Cobreloa      0-0   
9                      Segunda División         San Antonio Unido      1-1   
10                      Coupe Nationale             CR Belouizdad      0-0   
11      

,Liga,Local,Visitante,Marcador,Estado,Tarjetas Rojas (Local),Tarjetas Rojas (Visitante),Minuto
0,Tercera División RFEF - Group 12,Ibarra,Marino,1-1,First Half,0,0,37
1,Tercera División RFEF - Group 12,Unión Viera,Arucas,0-0,First Half,0,0,37
2,Tercera División RFEF - Group 12,UD San Fernando,Villa Santa Brígida,2-0,First Half,0,0,37
3,Division Intermedia,Independiente F.b.c.,Encarnación,1-0,First Half,0,0,38
4,Division Profesional - Apertura,Libertad Asuncion,2 de Mayo,0-0,First Half,0,0,23
5,Primera División,Estudiantes de Merida FC,Deportivo Tachira FC,0-0,First Half,0,0,9
6,Liga Pro,Aucas,Independiente del Valle,0-0,First Half,0,0,6
7,Primera División,Union Espanola,U. Catolica,0-1,First Half,0,0,24
8,Primera B,Cobreloa,Antofagasta,0-0,First Half,0,0,22
9,Segunda División,San Antonio Unido,Real San Joaquín,1-1,First Half,0,0,40


In [1]:
import requests

url = "http://site.api.espn.com/apis/site/v2/sports/soccer/eng.1/scoreboard"

response = requests.get(url)

if response.status_code == 200:
    data = response.json()
    
    for event in data["events"]:
        home_team = event["competitions"][0]["competitors"][0]["team"]["displayName"]
        away_team = event["competitions"][0]["competitors"][1]["team"]["displayName"]
        home_score = event["competitions"][0]["competitors"][0]["score"]
        away_score = event["competitions"][0]["competitors"][1]["score"]
        status = event["status"]["type"]["description"]  # Puede ser "In Progress", "Final", etc.
        
        print(f"{home_team} {home_score} - {away_score} {away_team} ({status})")
else:
    print(f"Error: {response.status_code}")


Arsenal 0 - 0 Fulham (Scheduled)
Wolverhampton Wanderers 0 - 0 West Ham United (Scheduled)
Nottingham Forest 0 - 0 Manchester United (Scheduled)


In [2]:
import requests
import time

url = "http://site.api.espn.com/apis/site/v2/sports/soccer/eng.1/scoreboard"

while True:
    response = requests.get(url)

    if response.status_code == 200:
        data = response.json()
        print("\n--- Partidos en vivo ---")
        
        for event in data["events"]:
            status = event["status"]["type"]["description"]  # "In Progress" = Partido en vivo
            if status == "In Progress":
                home_team = event["competitions"][0]["competitors"][0]["team"]["displayName"]
                away_team = event["competitions"][0]["competitors"][1]["team"]["displayName"]
                home_score = event["competitions"][0]["competitors"][0]["score"]
                away_score = event["competitions"][0]["competitors"][1]["score"]
                minute = event["status"]["displayClock"]  # Minuto actual del partido
                
                print(f"{home_team} {home_score} - {away_score} {away_team} | Minuto: {minute}")
    
    else:
        print("Error al obtener los datos.")
    
    time.sleep(10)  # Actualiza cada 10 segundos


--- Partidos en vivo ---

--- Partidos en vivo ---

--- Partidos en vivo ---

--- Partidos en vivo ---

--- Partidos en vivo ---

--- Partidos en vivo ---

--- Partidos en vivo ---

--- Partidos en vivo ---


KeyboardInterrupt: 

In [9]:
import requests

url = "https://api.sofascore.com/api/v1/sport/football/events/live"
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36"
}

response = requests.get(url, headers=headers)

if response.status_code == 200:
    data = response.json()
    
    for match in data["events"]:
        home_team = match["homeTeam"]["name"]
        away_team = match["awayTeam"]["name"]
        home_score = match["homeScore"]["current"]
        away_score = match["awayScore"]["current"]
        minute = match["status"]["minute"]
        
        print(f"{home_team} {home_score} - {away_score} {away_team} | Minuto: {minute}")

else:
    print(f"Error {response.status_code}: No se pudo obtener los datos.")





KeyError: 'minute'

In [ ]:
for match in data["events"]:
    home_team = match["homeTeam"]["name"]
    away_team = match["awayTeam"]["name"]
    home_score = match["homeScore"]["current"]
    away_score = match["awayScore"]["current"]
    minute = match["status"]["minute"]
    
    # Buscar estadísticas de tiros de esquina
    try:
        corners = next(stat for stat in match["statistics"] if stat["name"] == "Corners")["value"]
    except:
        corners = "No disponible"

    print(f"{home_team} {home_score} - {away_score} {away_team} | Minuto: {minute} | Tiros de esquina: {corners}")

In [10]:
import requests
import json

url = "https://api.sofascore.com/api/v1/sport/football/events/live"
headers = {"User-Agent": "Mozilla/5.0"}

response = requests.get(url, headers=headers)

if response.status_code == 200:
    data = response.json()
    
    # Guardar el JSON en un archivo para analizarlo
    with open("sofascore_data.json", "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=4)
    
    print("Datos guardados en sofascore_data.json")
else:
    print(f"Error {response.status_code}: No se pudo obtener los datos.")


Datos guardados en sofascore_data.json


In [11]:
import time
import requests

url = "https://api.sofascore.com/api/v1/sport/football/events/live"
headers = {
    "User-Agent": "Mozilla/5.0"
}

response = requests.get(url, headers=headers)

if response.status_code == 200:
    data = response.json()
    
    for match in data["events"]:
        home_team = match["homeTeam"]["name"]
        away_team = match["awayTeam"]["name"]
        home_score = match["homeScore"]["current"]
        away_score = match["awayScore"]["current"]

        # Calcular el minuto transcurrido
        current_time = int(time.time())  # Hora actual en segundos
        start_timestamp = match["startTimestamp"]  # Hora de inicio del partido
        elapsed_time = current_time - start_timestamp  # Tiempo transcurrido en segundos

        # Convertir el tiempo transcurrido a minutos
        elapsed_minutes = elapsed_time // 60
        
        print(f"{home_team} {home_score} - {away_score} {away_team} | Minuto: {elapsed_minutes}")
else:
    print(f"Error {response.status_code}: No se pudo obtener los datos.")


Sydney FC 1 - 2 Melbourne City | Minuto: 15
Albirex Niigata 1 - 1 Gamba Osaka | Minuto: 105
Kashiwa Reysol 1 - 0 Tokyo Verdy | Minuto: -15
Kyoto Sanga FC 1 - 0 Sanfrecce Hiroshima | Minuto: 210
Nagoya Grampus 2 - 1 Yokohama FC | Minuto: 120
Avispa Fukuoka 0 - 2 Machida Zelvia | Minuto: 45
Kashima Antlers 1 - 0 Vissel Kobe | Minuto: -30
Melbourne Victory 0 - 0 Adelaide United | Minuto: 85
FC Seoul 1 - 1 Daegu FC | Minuto: -45
Ansan Greeners FC 1 - 3 Hwaseong FC | Minuto: 165
Suwon Samsung Bluewings 1 - 2 Jeonnam Dragons | Minuto: -30
Melbourne Victory Youth 0 - 2 Green Gully | Minuto: -60
FC Bulleen Lions 1 - 2 Preston Lions FC | Minuto: 45
Mounties Wanderers U20 1 - 0 Macarthur Rams U20 | Minuto: -60
University of NSW 0 - 0 Inter Lions FC | Minuto: 150
Bankstown City Lions FC 0 - 0 Blacktown Spartans | Minuto: -225
SD Raiders 0 - 1 Canterbury Bankstown FC | Minuto: 0
South Coast Flame FC 1 - 1 Inner West Hawks FC | Minuto: -105
St George City U20 1 - 1 St George Saints FC U20 | Minuto: